In [ ]:
import argparse
import sys
import os

import gpxpy
import gpxpy.gpx
from typing import List, Tuple
from ..model.data import TrackPoint, Waypoint

def parse_gpx(file_path: str) -> Tuple[List[TrackPoint], List[Waypoint]]:
    """
    Parses a GPX file and returns a list of TrackPoints (with cumulative distance)
    and a list of Waypoints.
    """
    with open(file_path, 'r') as gpx_file:
        gpx = gpxpy.parse(gpx_file)

    track_points: List[TrackPoint] = []
    waypoints: List[Waypoint] = []
    
    # Process Tracks
    cumulative_distance = 0.0
    previous_point = None

    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                
                # Calculate distance from previous point
                if previous_point:
                    dist = point.distance_2d(previous_point)
                    if dist:
                         cumulative_distance += dist
                
                track_points.append(TrackPoint(
                    lat=point.latitude,
                    lon=point.longitude,
                    ele=point.elevation,
                    distance_from_start=cumulative_distance
                ))
                previous_point = point

    # Process Waypoints
    # Note: gpxpy parses top-level waypoints.
    # We map them to the track distance in a later step (pacer logic), 
    # here we just extract them.
    for wpt in gpx.waypoints:
        waypoints.append(Waypoint(
            name=wpt.name if wpt.name else f"Waypoint {wpt.latitude},{wpt.longitude}",
            lat=wpt.latitude,
            lon=wpt.longitude,
            distance_from_start=0.0 # To be calculated relative to track
        ))

    return track_points, waypoints


def map_waypoints_to_track(
    track_points: List[TrackPoint], 
    waypoints: List[Waypoint],
    off_route_threshold_m: float = 100.0
) -> Tuple[List[Waypoint], List[str]]:
    """
    Map waypoints to their nearest track points and assign distance_from_start.
    Waypoints are sorted by their distance along the track.
    
    Returns:
        Tuple of (mapped_waypoints, warnings) where warnings contains messages
        about any waypoints that are more than off_route_threshold_m from the track.
    """
    from ..lib.geo_utils import haversine_distance
    
    if not track_points or not waypoints:
        return [], []
    
    mapped_waypoints: List[Waypoint] = []
    warnings: List[str] = []
    
    for wpt in waypoints:
        min_dist_m = float('inf')
        nearest_track_dist = 0.0
        nearest_tp = track_points[0]
        
        # Find the nearest track point using haversine distance
        for tp in track_points:
            dist_m = haversine_distance(wpt.lat, wpt.lon, tp.lat, tp.lon)
            if dist_m < min_dist_m:
                min_dist_m = dist_m
                nearest_track_dist = tp.distance_from_start
                nearest_tp = tp
        
        # Warn if waypoint is far from track
        if min_dist_m > off_route_threshold_m:
            warnings.append(
                f"Warning: '{wpt.name}' is {min_dist_m:.0f}m from the track "
                f"(mapped to {nearest_track_dist/1000:.1f}km)"
            )
        
        mapped_waypoints.append(Waypoint(
            name=wpt.name,
            lat=wpt.lat,
            lon=wpt.lon,
            distance_from_start=nearest_track_dist
        ))
    
    # Sort by distance along track
    mapped_waypoints.sort(key=lambda w: w.distance_from_start)
    
    return mapped_waypoints, warnings


import csv
import json
from typing import List, Optional
from ..model.data import TrackPoint, SplitSegment, PacingPlan

def create_distance_splits(points: List[TrackPoint], split_distance_meters: float) -> List[SplitSegment]:
    if not points:
        return []

    splits: List[SplitSegment] = []
    
    current_split_start_dist = points[0].distance_from_start
    current_split_gain = 0.0
    current_split_loss = 0.0
    
    # We maintain a 'virtual' previous point to handle splits that happen mid-segment
    prev_point = points[0]
    
    # Target for the first split
    target_dist = current_split_start_dist + split_distance_meters
    
    # Iterate from the second point
    idx = 1
    while idx < len(points):
        curr_point = points[idx]
        
        # While the current point is beyond the target split distance, we have one or more splits to close
        while curr_point.distance_from_start > target_dist:
            # Fraction of the segment covered by the split
            segment_len = curr_point.distance_from_start - prev_point.distance_from_start
            
            # Avoid division by zero
            if segment_len == 0:
                 # Should not happen if distances are monotonic, but safety check
                 target_dist += split_distance_meters
                 continue

            dist_into_segment = target_dist - prev_point.distance_from_start
            fraction = dist_into_segment / segment_len
            
            # Interpolate elevation at target
            ele_diff = (curr_point.ele - prev_point.ele) if (curr_point.ele is not None and prev_point.ele is not None) else 0
            interpolated_ele = (prev_point.ele + (ele_diff * fraction)) if prev_point.ele is not None else None
            
            # Calculate gain/loss for this partial segment
            if prev_point.ele is not None and interpolated_ele is not None:
                diff = interpolated_ele - prev_point.ele
                if diff > 0:
                    current_split_gain += diff
                else:
                    current_split_loss += abs(diff)
            
            # Create the split
            splits.append(SplitSegment(
                start_distance=current_split_start_dist,
                end_distance=target_dist,
                length=split_distance_meters,
                elevation_gain=current_split_gain,
                elevation_loss=current_split_loss,
                name=f"{target_dist/1000:.1f} km" # Placeholder naming
            ))
            
            # Prepare for next split
            current_split_start_dist = target_dist
            current_split_gain = 0.0
            current_split_loss = 0.0
            target_dist += split_distance_meters
            
            # The 'previous point' for the next iteration is now our interpolated point
            # We construct a temporary point
            prev_point = TrackPoint(
                lat=0, # Lat/Lon not strictly needed for elevation/distance logic here, could interpolate if needed
                lon=0,
                ele=interpolated_ele,
                distance_from_start=current_split_start_dist
            )
            
        # If we are here, curr_point is within the current split (or equal to target if we handled loop exactly)
        # Add full segment gain/loss to current split
        if prev_point.ele is not None and curr_point.ele is not None:
            diff = curr_point.ele - prev_point.ele
            if diff > 0:
                current_split_gain += diff
            else:
                current_split_loss += abs(diff)
        
        prev_point = curr_point
        idx += 1
        
    # Handle remainder split if there's any distance left
    if prev_point.distance_from_start > current_split_start_dist:
        length = prev_point.distance_from_start - current_split_start_dist
        if length > 0.1: # Avoid microscopic floating point remainders
            splits.append(SplitSegment(
                start_distance=current_split_start_dist,
                end_distance=prev_point.distance_from_start,
                length=length,
                elevation_gain=current_split_gain,
                elevation_loss=current_split_loss,
                name="Finish"
            ))

    return splits


def create_waypoint_splits(
    points: List[TrackPoint], 
    waypoints: List['Waypoint']
) -> List[SplitSegment]:
    """
    Create splits between waypoints. Waypoints should already have 
    distance_from_start populated (use map_waypoints_to_track first).
    """
    from ..model.data import Waypoint  # Import here to avoid circular
    
    if not points:
        return []
    
    total_distance = points[-1].distance_from_start
    
    # Build split boundaries: Start, waypoints, Finish
    boundaries: List[tuple[float, str]] = [(0.0, "Start")]
    
    for wpt in waypoints:
        # Only include waypoints that are within the track
        if 0 < wpt.distance_from_start < total_distance:
            boundaries.append((wpt.distance_from_start, wpt.name))
    
    boundaries.append((total_distance, "Finish"))
    
    # Remove duplicates (waypoints at same location)
    seen_distances: set[float] = set()
    unique_boundaries: List[tuple[float, str]] = []
    for dist, name in boundaries:
        if dist not in seen_distances:
            unique_boundaries.append((dist, name))
            seen_distances.add(dist)
    boundaries = unique_boundaries
    
    splits: List[SplitSegment] = []
    
    for i in range(len(boundaries) - 1):
        start_dist, start_name = boundaries[i]
        end_dist, end_name = boundaries[i + 1]
        
        # Calculate elevation for this segment
        gain = 0.0
        loss = 0.0
        prev_ele = None
        
        for pt in points:
            if pt.distance_from_start < start_dist:
                continue
            if pt.distance_from_start > end_dist:
                break
                
            if prev_ele is not None and pt.ele is not None:
                diff = pt.ele - prev_ele
                if diff > 0:
                    gain += diff
                else:
                    loss += abs(diff)
            
            if pt.ele is not None:
                prev_ele = pt.ele
        
        segment_name = f"{start_name} to {end_name}"
        
        splits.append(SplitSegment(
            start_distance=start_dist,
            end_distance=end_dist,
            length=end_dist - start_dist,
            elevation_gain=gain,
            elevation_loss=loss,
            name=segment_name
        ))
    
    return splits


def generate_csv(plan: PacingPlan, output_path: str):
    # Check if surface data is present
    has_surface = any(s.surface is not None for s in plan.splits)

    fieldnames = ["Segment Name"]
    if has_surface:
        fieldnames.append("Surface")
    fieldnames.extend([
        "Distance (km)", 
        "Split Length (km)", 
        "Gain (m)", 
        "Loss (m)", 
        "Net Change (m)",
        "Cumulative Elevation (m)",
        "Grade (%)",
        # Planning columns (Empty placeholders for now)
        "Target Pace (min/km)",
        "Split Time (min)",
        "Station Delay (min)",
        "Arrival Time",
    ])
    
    with open(output_path, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        cumulative_elevation = 0.0
        for split in plan.splits:
            cumulative_elevation += split.elevation_gain - split.elevation_loss
            row = {
                "Segment Name": split.name,
                "Distance (km)": f"{split.end_distance / 1000:.2f}",
                "Split Length (km)": f"{split.length / 1000:.2f}",
                "Gain (m)": f"{split.elevation_gain:.0f}",
                "Loss (m)": f"{split.elevation_loss:.0f}",
                "Net Change (m)": f"{split.net_change:.0f}",
                "Cumulative Elevation (m)": f"{cumulative_elevation:.0f}",
                "Grade (%)": f"{split.grade:.1f}",
                "Target Pace (min/km)": "",
                "Split Time (min)": "",
                "Station Delay (min)": "",
                "Arrival Time": "",
            }
            if has_surface:
                row["Surface"] = split.surface or ""
            writer.writerow(row)


def generate_json(plan: PacingPlan, output_path: str):
    has_surface = any(s.surface is not None for s in plan.splits)

    splits_data: list[dict] = []
    cumulative_elevation = 0.0

    for split in plan.splits:
        cumulative_elevation += split.elevation_gain - split.elevation_loss
        entry: dict = {
            "segment_name": split.name,
            "distance_km": round(split.end_distance / 1000, 2),
            "split_length_km": round(split.length / 1000, 2),
            "gain_m": round(split.elevation_gain),
            "loss_m": round(split.elevation_loss),
            "net_change_m": round(split.net_change),
            "cumulative_elevation_m": round(cumulative_elevation),
            "grade_pct": round(split.grade, 1),
        }
        if has_surface:
            entry["surface"] = split.surface or ""
        splits_data.append(entry)

    output = {
        "metadata": plan.metadata,
        "splits": splits_data,
    }

    with open(output_path, 'w') as f:
        json.dump(output, f, indent=2)

from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

@dataclass
class TrackPoint:
    lat: float
    lon: float
    ele: Optional[float] = None
    distance_from_start: float = 0.0

@dataclass
class Waypoint:
    name: str
    lat: float
    lon: float
    distance_from_start: float = 0.0

@dataclass
class SplitSegment:
    start_distance: float
    end_distance: float
    length: float
    elevation_gain: float
    elevation_loss: float
    name: Optional[str] = None
    surface: Optional[str] = None
    
    @property
    def grade(self) -> float:
        if self.length == 0:
            return 0.0
        return ((self.elevation_gain - self.elevation_loss) / self.length) * 100

    @property
    def net_change(self) -> float:
        return self.elevation_gain - self.elevation_loss

@dataclass
class PacingPlan:
    metadata: Dict[str, Any]
    splits: List['SplitSegment'] = field(default_factory=list)


def main():
    parser = argparse.ArgumentParser(description="Generate pacing plan from GPX")
    parser.add_argument("input_file", help="Path to GPX file")
    parser.add_argument("-o", "--output", help="Output CSV file path")
    parser.add_argument("-m", "--split-mode", choices=["distance", "waypoint"], default="distance", help="Split mode")
    parser.add_argument("-d", "--split-dist", type=float, default=1.0, help="Split distance")
    parser.add_argument("-u", "--unit", choices=["km", "mi"], default="km", help="Unit for distance")
    parser.add_argument("--surface", action="store_true", help="Query OpenStreetMap for surface type per split (requires internet)")
    parser.add_argument("-f", "--format", choices=["csv", "json"], default="csv", help="Output format (default: csv)")
    
    args = parser.parse_args()
    
    # Validate input
    if not os.path.exists(args.input_file):
        print(f"Error: Input file '{args.input_file}' not found.")
        sys.exit(1)
        
    output_path = args.output
    if not output_path:
        base_name = os.path.splitext(args.input_file)[0]
        ext = "json" if args.format == "json" else "csv"
        output_path = f"{base_name}_pacing.{ext}"
        
    # 1. Parse GPX
    try:
        track_points, waypoints = parse_gpx(args.input_file)
    except Exception as e:
        print(f"Error parsing GPX: {e}")
        sys.exit(1)
        
    if not track_points:
        print("Error: No track points found in GPX.")
        sys.exit(1)

    # 2. Calculate Splits
    splits = []
    if args.split_mode == "distance":
        # Convert unit to meters
        dist_meters = args.split_dist * 1000 if args.unit == "km" else args.split_dist * 1609.34
        splits = create_distance_splits(track_points, dist_meters)
    else:
        # Waypoint mode
        mapped_waypoints, warnings = map_waypoints_to_track(track_points, waypoints)
        for warning in warnings:
            print(warning, file=sys.stderr)
        splits = create_waypoint_splits(track_points, mapped_waypoints)

    # 2b. Surface Detection (optional)
    if args.surface:
        from ..services.surface_service import detect_surfaces
        splits = detect_surfaces(splits, track_points)
        
    # 3. Generate Plan
    total_dist = track_points[-1].distance_from_start
    total_gain = sum(s.elevation_gain for s in splits)
    total_loss = sum(s.elevation_loss for s in splits)
    
    plan = PacingPlan(
        metadata={"filename": args.input_file, "total_dist": total_dist},
        splits=splits
    )
    
    # 4. Write Output
    try:
        if args.format == "json":
            generate_json(plan, output_path)
        else:
            generate_csv(plan, output_path)
        print(f"Successfully generated pacing plan: {output_path}")
    except Exception as e:
        print(f"Error writing output: {e}")
        sys.exit(1)
